In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
events_path = r"C:\Documents\Data analysis\Project 4 - Retail\events.csv"

In [6]:
event_sample = pd.read_csv(events_path, nrows=10)
event_sample

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
5,1433224086234,972639,view,22556,NaN
6,1433221923240,810725,view,443030,NaN
7,1433223291897,794181,view,439202,NaN
8,1433220899221,824915,view,428805,NaN
9,1433221204592,339335,view,82389,NaN


In [7]:
events = pd.read_csv(events_path)

In [8]:
events.shape

(2756101, 5)

In [9]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB


In [10]:
events['event'].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [11]:
events['visitorid'].nunique()

1407580

In [12]:
events['itemid'].nunique()

235061

In [13]:
events['transactionid'].nunique()

17672

In [14]:
events.isnull().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [15]:
events.duplicated().sum()

np.int64(460)

In [16]:
events['datetime']=pd.to_datetime(events['timestamp'],unit='ms')

In [17]:
events[['timestamp','datetime']].head()

,timestamp,datetime
0,1433221332117,2015-06-02 05:02:12.117
1,1433224214164,2015-06-02 05:50:14.164
2,1433221999827,2015-06-02 05:13:19.827
3,1433221955914,2015-06-02 05:12:35.914
4,1433221337106,2015-06-02 05:02:17.106


In [18]:
events['datetime'].min(), events['datetime'].max()

(Timestamp('2015-05-03 03:00:04.384000'),
 Timestamp('2015-09-18 02:59:47.788000'))

In [19]:
events['event'].value_counts(normalize=True)*100

event
view           96.669607
addtocart       2.515583
transaction     0.814810
Name: proportion, dtype: float64

In [20]:
events[events.duplicated(keep=False)].sort_values(['visitorid','timestamp']).head(20)

,timestamp,visitorid,event,itemid,transactionid,datetime
1041548,1440547735041,745,addtocart,246164,NaN,2015-08-26 00:08:55.041
1051024,1440547735041,745,addtocart,246164,NaN,2015-08-26 00:08:55.041
301904,1434501277715,3926,addtocart,323347,NaN,2015-06-17 00:34:37.715
319376,1434501277715,3926,addtocart,323347,NaN,2015-06-17 00:34:37.715
113491,1433611737367,10162,addtocart,146661,NaN,2015-06-06 17:28:57.367
124035,1433611737367,10162,addtocart,146661,NaN,2015-06-06 17:28:57.367
1965082,1432707397977,12046,addtocart,205663,NaN,2015-05-27 06:16:37.977
1983425,1432707397977,12046,addtocart,205663,NaN,2015-05-27 06:16:37.977
1166547,1441151647762,12560,addtocart,272976,NaN,2015-09-01 23:54:07.762
1167841,1441151647762,12560,addtocart,272976,NaN,2015-09-01 23:54:07.762


In [21]:
pd.crosstab(events['event'],events['transactionid'].isna())

transactionid,False,True
event,,
addtocart,0,69332
transaction,22457,0
view,0,2664312


In [22]:
events[events['event']=='transaction']['transactionid'].isna().sum()

np.int64(0)

In [23]:
visitor_activity = (
    events.groupby('visitorid')
    .agg(
        total_events=('event', 'size'),
        unique_items=('itemid', 'nunique'),
        views=('event', lambda x: (x == 'view').sum()),
        add_to_cart=('event', lambda x: (x == 'addtocart').sum()),
        transactions=('event', lambda x: (x == 'transaction').sum())
    )
    .reset_index()
)

visitor_activity.head()

,visitorid,total_events,unique_items,views,add_to_cart,transactions
0,0,3,3,3,0,0
1,1,1,1,1,0,0
2,2,8,4,8,0,0
3,3,1,1,1,0,0
4,4,1,1,1,0,0


In [24]:
visitor_activity.describe()

,visitorid,total_events,unique_items,views,add_to_cart,transactions
count,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06
mean,7.037895e+05,1.958042e+00,1.524019e+00,1.892832e+00,4.925617e-02,1.595433e-02
std,4.063335e+05,1.258049e+01,7.143724e+00,1.099370e+01,1.165057e+00,8.260909e-01
min,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.518948e+05,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
50%,7.037895e+05,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
75%,1.055684e+06,2.000000e+00,1.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00
max,1.407579e+06,7.757000e+03,3.814000e+03,6.479000e+03,7.190000e+02,5.590000e+02


In [25]:
funnel ={
    'Viewed Product':events.loc[events['event']=='view','visitorid'].nunique(),
    'Added to Cart':events.loc[events['event']=='addtocart','visitorid'].nunique(),
    'Purchased':events.loc[events['event']=='transaction','visitorid'].nunique()
}
funnel

{'Viewed Product': 1404179, 'Added to Cart': 37722, 'Purchased': 11719}

In [26]:
funnel_df=pd.DataFrame(funnel.items(),columns=['Stage','Unique Visitors'])
funnel_df

,Stage,Unique Visitors
0,Viewed Product,1404179
1,Added to Cart,37722
2,Purchased,11719


In [27]:
funnel_df['Conversion_Rate']=(
    funnel_df['Unique Visitors']/funnel_df['Unique Visitors'].iloc[0]*100)
funnel_df

,Stage,Unique Visitors,Conversion_Rate
0,Viewed Product,1404179,100.00000
1,Added to Cart,37722,2.68641
2,Purchased,11719,0.83458


In [28]:
funnel_df['Drop_off_Rate']=(
    1- funnel_df['Unique Visitors'].shift(-1)/funnel_df['Unique Visitors'])*100
funnel_df

,Stage,Unique Visitors,Conversion_Rate,Drop_off_Rate
0,Viewed Product,1404179,100.00000,97.313590
1,Added to Cart,37722,2.68641,68.933249
2,Purchased,11719,0.83458,NaN


In [29]:
funnel_df.loc[funnel_df['Drop_off_Rate'].idxmax()]

Stage              Viewed Product
Unique Visitors           1404179
Conversion_Rate             100.0
Drop_off_Rate            97.31359
Name: 0, dtype: object

In [30]:
cart_events=events[events['event']=='addtocart'].copy()

In [31]:
cart_events.shape

(69332, 6)

In [32]:
cart_events.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
17,1433223236124,287857,addtocart,5206,NaN,2015-06-02 05:33:56.124
19,1433221078505,158090,addtocart,10572,NaN,2015-06-02 04:57:58.505
63,1433223543021,1193904,addtocart,255275,NaN,2015-06-02 05:39:03.021
112,1433221941632,599528,addtocart,356475,NaN,2015-06-02 05:12:21.632
179,1433220880956,105775,addtocart,312728,NaN,2015-06-02 04:54:40.956


In [33]:
transactions=events[events['event']=='transaction'].copy()

In [34]:
transactions.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
130,1433222276276,599528,transaction,356475,4000.0,2015-06-02 05:17:56.276
304,1433193500981,121688,transaction,15335,11117.0,2015-06-01 21:18:20.981
418,1433193915008,552148,transaction,81345,5444.0,2015-06-01 21:25:15.008
814,1433176736375,102019,transaction,150318,13556.0,2015-06-01 16:38:56.375
843,1433174518180,189384,transaction,310791,7244.0,2015-06-01 16:01:58.180


In [35]:
transactions.shape

(22457, 6)

In [36]:
cart_events[['visitorid', 'itemid', 'timestamp', 'datetime']].head()

,visitorid,itemid,timestamp,datetime
17,287857,5206,1433223236124,2015-06-02 05:33:56.124
19,158090,10572,1433221078505,2015-06-02 04:57:58.505
63,1193904,255275,1433223543021,2015-06-02 05:39:03.021
112,599528,356475,1433221941632,2015-06-02 05:12:21.632
179,105775,312728,1433220880956,2015-06-02 04:54:40.956


In [37]:
transactions[['visitorid', 'itemid', 'transactionid', 'timestamp', 'datetime']].head()

,visitorid,itemid,transactionid,timestamp,datetime
130,599528,356475,4000.0,1433222276276,2015-06-02 05:17:56.276
304,121688,15335,11117.0,1433193500981,2015-06-01 21:18:20.981
418,552148,81345,5444.0,1433193915008,2015-06-01 21:25:15.008
814,102019,150318,13556.0,1433176736375,2015-06-01 16:38:56.375
843,189384,310791,7244.0,1433174518180,2015-06-01 16:01:58.180


In [38]:
print("Add-to-cart events:", len(cart_events))
print("Transaction events:", len(transactions))

Add-to-cart events: 69332
Transaction events: 22457


In [39]:
print("Unique visitors adding to cart:",cart_events['visitorid'].nunique())
print("Unique visitors purchasing:",transactions['visitorid'].nunique())

Unique visitors adding to cart: 37722
Unique visitors purchasing: 11719


In [40]:
transactions[['visitorid','itemid','transactionid']].head(10)

,visitorid,itemid,transactionid
130,599528,356475,4000.0
304,121688,15335,11117.0
418,552148,81345,5444.0
814,102019,150318,13556.0
843,189384,310791,7244.0
1214,350566,54058,8796.0
1215,350566,284871,8796.0
1234,404403,150100,5216.0
1350,505565,243566,11713.0
1512,945184,245400,2415.0


In [41]:
cart_events=cart_events.sort_values(
    ['visitorid','itemid','timestamp']).copy()
transactions=transactions.sort_values(
    ['visitorid','itemid','timestamp']).copy()

In [42]:
cart_events.columns

Index(['timestamp', 'visitorid', 'event', 'itemid', 'transactionid',
       'datetime'],
      dtype='object')

In [43]:
cart_events = cart_events.sort_values('timestamp').reset_index(drop=True)

transactions = transactions.sort_values('timestamp').reset_index(drop=True)

In [44]:
print(cart_events['timestamp'].is_monotonic_increasing)
print(transactions['timestamp'].is_monotonic_increasing)

True
True


In [45]:
cart_purchase_match = pd.merge_asof(
    cart_events,
    transactions,
    on='timestamp',
    by=['visitorid', 'itemid'],
    direction='forward',
    allow_exact_matches=False,
    suffixes=('_cart', '_transaction')
)

In [46]:
cart_purchase_match.head()

,timestamp,visitorid,event_cart,itemid,transactionid_cart,datetime_cart,event_transaction,transactionid_transaction,datetime_transaction
0,1430622004384,693516,addtocart,297662,NaN,2015-05-03 03:00:04.384,NaN,NaN,NaT
1,1430622029427,693516,addtocart,297662,NaN,2015-05-03 03:00:29.427,NaN,NaN,NaT
2,1430622085008,979664,addtocart,338222,NaN,2015-05-03 03:01:25.008,NaN,NaN,NaT
3,1430622240603,260113,addtocart,125751,NaN,2015-05-03 03:04:00.603,NaN,NaN,NaT
4,1430622602006,319455,addtocart,342530,NaN,2015-05-03 03:10:02.006,NaN,NaN,NaT


In [47]:
cart_purchase_match.shape

(69332, 9)

In [48]:
cart_purchase_match['transactionid_transaction'].notna().sum()

np.int64(21404)

In [49]:
cart_purchase_match['transactionid_transaction'].notna().mean() * 100

np.float64(30.871747533606413)

In [50]:
cart_purchase_match['purchased'] =cart_purchase_match['transactionid_transaction'].notna()
cart_purchase_match['purchased'].value_counts()

purchased
False    47928
True     21404
Name: count, dtype: int64

In [51]:
cart_purchase_match.head()

,timestamp,visitorid,event_cart,itemid,transactionid_cart,datetime_cart,event_transaction,transactionid_transaction,datetime_transaction,purchased
0,1430622004384,693516,addtocart,297662,NaN,2015-05-03 03:00:04.384,NaN,NaN,NaT,False
1,1430622029427,693516,addtocart,297662,NaN,2015-05-03 03:00:29.427,NaN,NaN,NaT,False
2,1430622085008,979664,addtocart,338222,NaN,2015-05-03 03:01:25.008,NaN,NaN,NaT,False
3,1430622240603,260113,addtocart,125751,NaN,2015-05-03 03:04:00.603,NaN,NaN,NaT,False
4,1430622602006,319455,addtocart,342530,NaN,2015-05-03 03:10:02.006,NaN,NaN,NaT,False


In [52]:
cart_purchase_match['abandoned']=(~cart_purchase_match['purchased'])

In [53]:
cart_purchase_match['abandoned'].value_counts()

abandoned
True     47928
False    21404
Name: count, dtype: int64

In [54]:
cart_purchase_match['abandoned'].mean() * 100

np.float64(69.12825246639358)

In [55]:
cart_purchase_match[['visitorid','itemid']].drop_duplicates().shape

(62025, 2)

In [56]:
cart_purchase_match.groupby(['visitorid','itemid'])['purchased'].max().value_counts()

purchased
False    43022
True     19003
Name: count, dtype: int64

In [57]:
cart_level=(
    cart_purchase_match.groupby(['visitorid','itemid'])
    .agg(cart_time=('datetime_cart','min'),purchased=('purchased','max')
        )
)

In [58]:
cart_level['abandoned'] = ~cart_level['purchased']

In [59]:
cart_level=cart_level.reset_index()

In [60]:
cart_level.head()

,visitorid,itemid,cart_time,purchased,abandoned
0,6,65273,2015-08-30 06:03:48.202,False,True
1,150,452955,2015-06-07 23:30:18.230,False,True
2,155,41882,2015-09-11 06:17:44.613,False,True
3,155,181405,2015-09-11 06:12:47.871,False,True
4,155,224623,2015-09-11 06:19:00.395,False,True


In [61]:
cart_level.columns

Index(['visitorid', 'itemid', 'cart_time', 'purchased', 'abandoned'], dtype='object')

In [62]:
cart_level.shape

(62025, 5)

In [63]:
cart_level['abandoned'].value_counts(normalize=True) * 100

abandoned
True     69.362354
False    30.637646
Name: proportion, dtype: float64

## KPI SUMMARY 

In [64]:
total_cart_items=len(cart_level)
purchased_items=cart_level['purchased'].sum()
abandoned_items=cart_level['abandoned'].sum()

print("Total cart opportunities:", total_cart_items)
print("Purchased:", purchased_items)
print("Abandoned:",abandoned_items)
print("Purchase conversion rate:", purchased_items/total_cart_items *100)
print("Observed non-conversion rate:", abandoned_items/total_cart_items *100)

Total cart opportunities: 62025
Purchased: 19003
Abandoned: 43022
Purchase conversion rate: 30.637646110439338
Observed non-conversion rate: 69.36235388956065


In [68]:
category_tree = pd.read_csv(r"C:\Documents\Data analysis\Project 4 - Retail\category_tree.csv")

In [69]:
category_tree.head()

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [70]:
category_tree.shape

(1669, 2)

In [71]:
category_tree.isnull().sum()

categoryid     0
parentid      25
dtype: int64

In [73]:
item_prop1 = pd.read_csv(r"C:\Documents\Data analysis\Project 4 - Retail\item_properties_part1.csv")
item_prop2 = pd.read_csv(r"C:\Documents\Data analysis\Project 4 - Retail\item_properties_part2.csv")

In [74]:
item_prop1.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


In [75]:
item_prop1.columns
item_prop2.columns

Index(['timestamp', 'itemid', 'property', 'value'], dtype='object')

In [76]:
item_prop1['property'].nunique()
item_prop2['property'].nunique()

1094

In [77]:
item_prop1['property'].value_counts().head(20)

property
888           1629817
790            970800
available      817387
categoryid     426305
6              343207
283            323681
776            311654
678            261829
364            256340
202            242984
839            226921
159            226502
917            226437
764            226242
112            226102
227            188209
698            157281
451            142388
663            131331
962            128976
Name: count, dtype: int64

In [78]:
item_prop2['property'].value_counts().head(20)

property
888           1370581
790            819716
available      686252
categoryid     361909
6              288264
283            273738
776            262566
364            220146
678            220137
202            205954
112            190951
764            190811
917            190790
159            190551
839            190318
227            159283
698            132568
451            122028
962            110396
663            109482
Name: count, dtype: int64

In [79]:
item_cat1=item_prop1[
    item_prop1['property']=='categoryid'
].copy()
item_cat2=item_prop2[
    item_prop2['property']=='categoryid'
].copy()

In [80]:
item_cat1.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
140,1432436400000,281245,categoryid,1277
151,1435460400000,35575,categoryid,1059
189,1437274800000,8313,categoryid,1147
197,1437879600000,55102,categoryid,47


In [81]:
item_category=pd.concat(
    [item_cat1,item_cat2],
    ignore_index=True
)

In [82]:
item_category.shape

(788214, 4)

In [83]:
item_category.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1432436400000,281245,categoryid,1277
2,1435460400000,35575,categoryid,1059
3,1437274800000,8313,categoryid,1147
4,1437879600000,55102,categoryid,47


In [84]:
item_category[['itemid','value']].duplicated().sum()

np.int64(345782)

In [85]:
item_category['itemid'].nunique()

417053

In [86]:
item_category.groupby('itemid')['value'].nunique().value_counts().sort_index()

value
1    393701
2     21373
3      1931
4        48
Name: count, dtype: int64

In [87]:
multi_category_items=(
    item_category.groupby('itemid')['value'].nunique().loc[lambda x:x>1])

In [88]:
multi_category_items.head()

itemid
25     2
94     2
130    2
149    2
168    2
Name: value, dtype: int64

In [89]:
cart_level.columns

Index(['visitorid', 'itemid', 'cart_time', 'purchased', 'abandoned'], dtype='object')

In [90]:
cart_items=cart_level[['itemid']].drop_duplicates()

In [91]:
print("Unique items in cart:",cart_items['itemid'].nunique())

Unique items in cart: 23903


In [92]:
print(
    "Item with category information:",
    cart_items['itemid'].isin(item_category['itemid']).sum()
)

Item with category information: 23458


In [93]:
print(
    "Percentage with category information:",
    cart_items['itemid'].isin(item_category['itemid']).mean()*100
)

Percentage with category information: 98.13830899887044


In [94]:
cart_item_categories=item_category[
    item_category['itemid'].isin(cart_items['itemid'])
].copy()
cart_item_categories.head()

,timestamp,itemid,property,value
30,1438484400000,313481,categoryid,1613
41,1438484400000,442141,categoryid,1277
60,1431226800000,38175,categoryid,196
61,1435460400000,425951,categoryid,1483
63,1438484400000,74371,categoryid,1613


In [95]:
print("Category records:", len(cart_item_categories))
print("Unique items:", cart_item_categories['itemid'].nunique())
print("Unique categories:", cart_item_categories['value'].nunique())

Category records: 48136
Unique items: 23458
Unique categories: 826


In [96]:
cart_item_categories.groupby('itemid')['value'].nunique().value_counts().sort_index()

value
1    21952
2     1401
3       94
4       11
Name: count, dtype: int64

In [97]:
item_category['category_datetime']=pd.to_datetime(
    item_category['timestamp'],unit='ms'
)

In [98]:
item_category[['timestamp','category_datetime']].head()

,timestamp,category_datetime
0,1435460400000,2015-06-28 03:00:00
1,1432436400000,2015-05-24 03:00:00
2,1435460400000,2015-06-28 03:00:00
3,1437274800000,2015-07-19 03:00:00
4,1437879600000,2015-07-26 03:00:00


In [99]:
print(item_category['category_datetime'].min())
print(item_category['category_datetime'].max())

2015-05-10 03:00:00
2015-09-13 03:00:00


In [100]:
category_map=item_category[
    ['itemid','category_datetime','value']
].copy()
category_map=category_map.rename(
    columns={'value':'categoryid'}
)

In [101]:
category_map.head()

,itemid,category_datetime,categoryid
0,460429,2015-06-28 03:00:00,1338
1,281245,2015-05-24 03:00:00,1277
2,35575,2015-06-28 03:00:00,1059
3,8313,2015-07-19 03:00:00,1147
4,55102,2015-07-26 03:00:00,47


In [102]:
cart_for_category = cart_level[['visitorid', 'itemid', 'cart_time', 'purchased', 'abandoned']].copy()

In [103]:
cart_for_category.head()

,visitorid,itemid,cart_time,purchased,abandoned
0,6,65273,2015-08-30 06:03:48.202,False,True
1,150,452955,2015-06-07 23:30:18.230,False,True
2,155,41882,2015-09-11 06:17:44.613,False,True
3,155,181405,2015-09-11 06:12:47.871,False,True
4,155,224623,2015-09-11 06:19:00.395,False,True


In [104]:
cart_for_category=cart_for_category.sort_values('cart_time').reset_index(drop=True)
category_map=category_map.sort_values('category_datetime').reset_index(drop=True)

In [105]:
cart_with_category=pd.merge_asof(
    cart_for_category,category_map,left_on='cart_time',right_on='category_datetime',
    by='itemid',direction='backward')

In [106]:
cart_with_category.head()

,visitorid,itemid,cart_time,purchased,abandoned,category_datetime,categoryid
0,693516,297662,2015-05-03 03:00:04.384,False,True,NaT,NaN
1,979664,338222,2015-05-03 03:01:25.008,False,True,NaT,NaN
2,260113,125751,2015-05-03 03:04:00.603,False,True,NaT,NaN
3,319455,342530,2015-05-03 03:10:02.006,False,True,NaT,NaN
4,345781,438400,2015-05-03 03:11:33.415,True,False,NaT,NaN


In [107]:
cart_with_category.shape

(62025, 7)

In [108]:
cart_with_category['categoryid'].isna().sum()

np.int64(10835)

In [109]:
item_category_mode=(
    item_category.groupby('itemid')['value']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

In [110]:
cart_with_category=cart_level.merge(
    item_category_mode,on='itemid',how='left')

In [111]:
cart_with_category.head()

,visitorid,itemid,cart_time,purchased,abandoned,value
0,6,65273,2015-08-30 06:03:48.202,False,True,342
1,150,452955,2015-06-07 23:30:18.230,False,True,577
2,155,41882,2015-09-11 06:17:44.613,False,True,769
3,155,181405,2015-09-11 06:12:47.871,False,True,769
4,155,224623,2015-09-11 06:19:00.395,False,True,769


In [112]:
cart_with_category.shape

(62025, 6)

In [113]:
cart_with_category['value'].isna().sum()

np.int64(791)

In [114]:
cart_with_category['value'].notna().mean()*100

np.float64(98.72470777912132)

In [115]:
cart_with_category=cart_with_category.rename(
    columns={'value':'categoryid'})

In [116]:
cart_with_category.head()

,visitorid,itemid,cart_time,purchased,abandoned,categoryid
0,6,65273,2015-08-30 06:03:48.202,False,True,342
1,150,452955,2015-06-07 23:30:18.230,False,True,577
2,155,41882,2015-09-11 06:17:44.613,False,True,769
3,155,181405,2015-09-11 06:12:47.871,False,True,769
4,155,224623,2015-09-11 06:19:00.395,False,True,769


In [117]:
cart_with_category['categoryid']=(
    cart_with_category['categoryid'].fillna('Unknown'))

In [118]:
cart_with_category['categoryid'].isna().sum()

np.int64(0)

In [119]:
category_analysis=(
    cart_with_category
    .groupby('categoryid')
    .agg(
        cart_opportunities=('itemid','count'),
        purchased=('purchased','sum'),
        abandoned=('abandoned','sum')
    ).reset_index()
)

In [120]:
category_analysis.head()

,categoryid,cart_opportunities,purchased,abandoned
0,0,52,12,40
1,1,344,68,276
2,1001,19,5,14
3,1002,97,26,71
4,1003,22,5,17


In [121]:
category_analysis['abandonment_rate']=(
    category_analysis['abandoned']/category_analysis['cart_opportunities']*100)

In [122]:
category_analysis=category_analysis.sort_values('abandonment_rate',ascending=False)

In [123]:
category_analysis.head(20)

,categoryid,cart_opportunities,purchased,abandoned,abandonment_rate
275,1506,6,0,6,100.0
624,652,1,0,1,100.0
750,932,4,0,4,100.0
337,1620,1,0,1,100.0
638,682,2,0,2,100.0
195,1351,2,0,2,100.0
357,1660,1,0,1,100.0
71,1128,1,0,1,100.0
371,1684,1,0,1,100.0
379,182,3,0,3,100.0


In [124]:
category_analysis_50 = category_analysis[
    category_analysis['cart_opportunities']>=50].copy()

In [125]:
category_analysis_50 = category_analysis_50.sort_values(
    'abandonment_rate',ascending=False)

In [126]:
category_analysis_50.head()

,categoryid,cart_opportunities,purchased,abandoned,abandonment_rate
91,1171,100,5,95,95.000000
274,1504,60,3,57,95.000000
37,1076,85,5,80,94.117647
540,499,250,15,235,94.000000
616,64,220,14,206,93.636364


In [127]:
category_analysis_50.shape

(278, 5)

In [128]:
cart_with_category['hour']=cart_with_category['cart_time'].dt.hour

In [129]:
cart_with_category[['cart_time','hour']].head()

,cart_time,hour
0,2015-08-30 06:03:48.202,6
1,2015-06-07 23:30:18.230,23
2,2015-09-11 06:17:44.613,6
3,2015-09-11 06:12:47.871,6
4,2015-09-11 06:19:00.395,6


In [130]:
hour_analysis=(
    cart_with_category.groupby('hour')
    .agg(
        cart_opportunities=('itemid','count'),
        purchased=('purchased','sum'),
        abandoned=('abandoned','sum')
    ).reset_index()
)

In [131]:
hour_analysis['abandonment_rate']=(
    hour_analysis['abandoned']/hour_analysis['cart_opportunities']*100)

In [132]:
hour_analysis.sort_values('hour')

,hour,cart_opportunities,purchased,abandoned,abandonment_rate
0,0,3249,945,2304,70.914127
1,1,3169,849,2320,73.209214
2,2,2970,908,2062,69.427609
3,3,2961,696,2265,76.494428
4,4,2826,602,2224,78.697806
5,5,2526,597,1929,76.365796
6,6,1620,351,1269,78.333333
7,7,926,212,714,77.105832
8,8,476,77,399,83.823529
9,9,352,53,299,84.943182


In [133]:
funnel_df

,Stage,Unique Visitors,Conversion_Rate,Drop_off_Rate
0,Viewed Product,1404179,100.00000,97.313590
1,Added to Cart,37722,2.68641,68.933249
2,Purchased,11719,0.83458,NaN


#USER SEGMENTATION

In [134]:
visitor_activity.head()

,visitorid,total_events,unique_items,views,add_to_cart,transactions
0,0,3,3,3,0,0
1,1,1,1,1,0,0
2,2,8,4,8,0,0
3,3,1,1,1,0,0
4,4,1,1,1,0,0


In [135]:
def classify_user(row):
    if row['transactions']>0:
        return 'Purchaser'
    elif row['add_to_cart']>0:
        return 'Cart Non-Converter'
    else:
        return 'Viewer Only'

visitor_activity['user_segment']=visitor_activity.apply(
    classify_user,axis=1)

In [136]:
visitor_activity['user_segment'].value_counts()

user_segment
Viewer Only           1368715
Cart Non-Converter      27146
Purchaser               11719
Name: count, dtype: int64

In [137]:
visitor_activity['user_segment'].value_counts(normalize=True).mul(100).round(2)

user_segment
Viewer Only           97.24
Cart Non-Converter     1.93
Purchaser              0.83
Name: proportion, dtype: float64

##COHORT ANALSYSIS 

In [138]:
events.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


In [139]:
events[['visitorid','datetime']].head()

,visitorid,datetime
0,257597,2015-06-02 05:02:12.117
1,992329,2015-06-02 05:50:14.164
2,111016,2015-06-02 05:13:19.827
3,483717,2015-06-02 05:12:35.914
4,951259,2015-06-02 05:02:17.106


In [140]:
first_activity=(
    events.groupby('visitorid')['datetime'].min().reset_index()
)
first_activity.columns=['visitorid','datetime']

In [141]:
first_activity.head()

,visitorid,datetime
0,0,2015-09-11 20:49:49.439
1,1,2015-08-13 17:46:06.444
2,2,2015-08-07 17:51:44.567
3,3,2015-08-01 07:10:35.296
4,4,2015-09-15 21:24:27.167


In [142]:
first_activity['cohort_month']=(first_activity['datetime'].dt.to_period('M'))

In [143]:
first_activity.head()

,visitorid,datetime,cohort_month
0,0,2015-09-11 20:49:49.439,2015-09
1,1,2015-08-13 17:46:06.444,2015-08
2,2,2015-08-07 17:51:44.567,2015-08
3,3,2015-08-01 07:10:35.296,2015-08
4,4,2015-09-15 21:24:27.167,2015-09


In [144]:
cohort_size=(
    first_activity
    .groupby('cohort_month')['visitorid'].nunique().reset_index(name='cohort_users'))

In [145]:
cohort_size

,cohort_month,cohort_users
0,2015-05,307574
1,2015-06,299189
2,2015-07,355765
3,2015-08,288328
4,2015-09,156724


In [146]:
events_cohort=events.merge(first_activity[['visitorid','cohort_month']],
                           on='visitorid',how='left')
events_cohort=events_cohort.sort_values('cohort_month',ascending=True)
events_cohort.head(20)

,timestamp,visitorid,event,itemid,transactionid,datetime,cohort_month
2007221,1432918745491,616878,view,241874,NaN,2015-05-29 16:59:05.491,2015-05
1581044,1431129228120,1400282,view,56947,NaN,2015-05-08 23:53:48.120,2015-05
1581043,1431132265749,453083,view,252398,NaN,2015-05-09 00:44:25.749,2015-05
1581042,1431109250985,1233519,view,459791,NaN,2015-05-08 18:20:50.985,2015-05
480074,1435169792201,152693,view,184465,NaN,2015-06-24 18:16:32.201,2015-05
1581041,1431109637548,968683,view,199819,NaN,2015-05-08 18:27:17.548,2015-05
480076,1435168475743,899584,view,223440,NaN,2015-06-24 17:54:35.743,2015-05
1581040,1431110423477,1361641,view,280620,NaN,2015-05-08 18:40:23.477,2015-05
1581039,1431111140199,869928,view,34212,NaN,2015-05-08 18:52:20.199,2015-05
1581038,1431109229190,1196757,view,345488,NaN,2015-05-08 18:20:29.190,2015-05


In [147]:
cohort_purchase=(
    events_cohort[events_cohort['event']=='transaction']
    .groupby('cohort_month')['visitorid'].nunique()
    .reset_index(name='purchasers')
)
cohort_purchase

,cohort_month,purchasers
0,2015-05,2947
1,2015-06,2712
2,2015-07,2764
3,2015-08,2233
4,2015-09,1063


In [148]:
cohort=cohort_size.merge(cohort_purchase,on='cohort_month',how='left')
cohort

,cohort_month,cohort_users,purchasers
0,2015-05,307574,2947
1,2015-06,299189,2712
2,2015-07,355765,2764
3,2015-08,288328,2233
4,2015-09,156724,1063


In [149]:
cohort['cohort_purchase_rate']=((cohort['purchasers']/cohort['cohort_users']).mul(100).round(2))

In [150]:
cohort

,cohort_month,cohort_users,purchasers,cohort_purchase_rate
0,2015-05,307574,2947,0.96
1,2015-06,299189,2712,0.91
2,2015-07,355765,2764,0.78
3,2015-08,288328,2233,0.77
4,2015-09,156724,1063,0.68


In [151]:
visitor_days=(
    events.assign(activity_date=events['datetime'].dt.date)
    .groupby('visitorid')['activity_date'].nunique()
    .reset_index(name='active_days')
)
visitor_days.head()

,visitorid,active_days
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1


In [152]:
events['visitorid'].nunique()

1407580

In [153]:
visitor_days['repeat_visitors']=visitor_days['active_days']>1

In [154]:
visitor_days['repeat_visitors'].value_counts()

repeat_visitors
False    1263811
True      143769
Name: count, dtype: int64

In [155]:
visitor_cohort=first_activity.merge(
    visitor_days,on='visitorid',how='left')
visitor_cohort.head()

,visitorid,datetime,cohort_month,active_days,repeat_visitors
0,0,2015-09-11 20:49:49.439,2015-09,1,False
1,1,2015-08-13 17:46:06.444,2015-08,1,False
2,2,2015-08-07 17:51:44.567,2015-08,1,False
3,3,2015-08-01 07:10:35.296,2015-08,1,False
4,4,2015-09-15 21:24:27.167,2015-09,1,False


In [156]:
cohort_repeat=(
    visitor_cohort
    .groupby('cohort_month')
    .agg(cohort_users=('visitorid','nunique'),repeat_visitors=('repeat_visitors','sum'))
    .reset_index()
)

In [157]:
cohort_repeat.head()

,cohort_month,cohort_users,repeat_visitors
0,2015-05,307574,43955
1,2015-06,299189,35889
2,2015-07,355765,32965
3,2015-08,288328,22811
4,2015-09,156724,8149


In [158]:
cohort_repeat['repeat_rate']=(
    cohort_repeat['repeat_visitors']/cohort_repeat['cohort_users']*100).round(2)

In [159]:
cohort_repeat
    

,cohort_month,cohort_users,repeat_visitors,repeat_rate
0,2015-05,307574,43955,14.29
1,2015-06,299189,35889,12.00
2,2015-07,355765,32965,9.27
3,2015-08,288328,22811,7.91
4,2015-09,156724,8149,5.20


In [160]:
cohort_overall=cohort_repeat.merge(cohort,on=['cohort_month','cohort_users'],how='left')

In [161]:
cohort_overall

,cohort_month,cohort_users,repeat_visitors,repeat_rate,purchasers,cohort_purchase_rate
0,2015-05,307574,43955,14.29,2947,0.96
1,2015-06,299189,35889,12.00,2712,0.91
2,2015-07,355765,32965,9.27,2764,0.78
3,2015-08,288328,22811,7.91,2233,0.77
4,2015-09,156724,8149,5.20,1063,0.68


In [167]:
user_segments = (
    visitor_activity
    .assign(
        user_segment=lambda df: np.select(
            [
                df['transactions'] > 0,
                df['add_to_cart'] > 0
            ],
            [
                'Purchaser',
                'Cart Non-Converter'
            ],
            default='Viewer Only'
        )
    )
)

user_segments['user_segment'].value_counts()

user_segments.head()

,visitorid,total_events,unique_items,views,add_to_cart,transactions,user_segment
0,0,3,3,3,0,0,Viewer Only
1,1,1,1,1,0,0,Viewer Only
2,2,8,4,8,0,0,Viewer Only
3,3,1,1,1,0,0,Viewer Only
4,4,1,1,1,0,0,Viewer Only


In [168]:
user_segment_summary = (
    user_segments.groupby('user_segment')
    .agg(
        users=('visitorid', 'count'),total_views=('views', 'sum'),
        total_add_to_cart=('add_to_cart', 'sum'),total_transactions=('transactions', 'sum')
    ).reset_index()
)
user_segment_summary

,user_segment,users,total_views,total_add_to_cart,total_transactions
0,Cart Non-Converter,27146,149126,41865,0
1,Purchaser,11719,180754,27467,22457
2,Viewer Only,1368715,2334432,0,0


##EXPORT TO EXCEL

In [172]:
file_path = r'C:\Documents\Data analysis\Project 4 - Retail\Analysis\RetailRocket_PowerBI_Data.xlsx'

with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    
    funnel_df.to_excel(writer, sheet_name='Funnel', index=False)
    category_analysis_50.to_excel(writer, sheet_name='Category', index=False)
    hour_analysis.to_excel(writer, sheet_name='Time_Analysis', index=False)
    user_segment_summary.to_excel(writer, sheet_name='User_Segments', index=False)
    cohort_overall.to_excel(writer, sheet_name='Cohort', index=False)

In [173]:
print(file_path)

C:\Documents\Data analysis\Project 4 - Retail\Analysis\RetailRocket_PowerBI_Data.xlsx
